## Step 1: Import Libraries and Generate Portfolio

In [2]:
import numpy as np
import tensorflow as tf
from tensor_engine import generate_synthetic_portfolio, deterministic_loss, TensorialRiskEngine

# Generate the same portfolio as minimum_example_deterministic.py
print("Generating portfolio with 10 assets, 1 event, 5 typologies, 20 curve points...")
v, u, C, x_grid, H, lamb = generate_synthetic_portfolio(
    n_assets=10,
    n_events=1,
    n_typologies=5,
    n_curve_points=20
)

print(f"\n✓ Portfolio generated successfully")
print(f"  - Exposure vector (v): {v.shape}")
print(f"  - Typology vector (u): {u.shape}")
print(f"  - Vulnerability matrix (C): {C.shape}")
print(f"  - Intensity grid (x_grid): {x_grid.shape}")
print(f"  - Hazard matrix (H): {H.shape}")

Generating portfolio with 10 assets, 1 event, 5 typologies, 20 curve points...

✓ Portfolio generated successfully
  - Exposure vector (v): (10,)
  - Typology vector (u): (10,)
  - Vulnerability matrix (C): (5, 20)
  - Intensity grid (x_grid): (20,)
  - Hazard matrix (H): (10, 1)


## Step 2: Display Portfolio Data

Let's examine the portfolio data in detail.

In [3]:
# Extract hazard intensities for the single event
h = H[:, 0]  # Shape: (10,) - intensity at each asset

print("="*80)
print("PORTFOLIO DATA")
print("="*80)

print("\n1. EXPOSURE VECTOR (v) - Replacement costs in $")
print("-" * 80)
for i in range(len(v)):
    print(f"   Asset {i}: ${v[i]:>12,.2f}")
print(f"\n   Total portfolio value: ${v.sum():,.2f}")

print("\n2. TYPOLOGY ASSIGNMENT (u) - Which vulnerability curve each asset uses")
print("-" * 80)
for i in range(len(u)):
    print(f"   Asset {i}: Typology {u[i]}")

print("\n3. HAZARD INTENSITIES (h) - Ground motion in g for Event 0")
print("-" * 80)
for i in range(len(h)):
    print(f"   Asset {i}: {h[i]:.4f} g")
print(f"\n   Intensity range: {h.min():.4f} g to {h.max():.4f} g")

print("\n4. INTENSITY GRID (x_grid) - Common intensity axis for all curves")
print("-" * 80)
print(f"   Number of points: {len(x_grid)}")
print(f"   Range: {x_grid.min():.4f} g to {x_grid.max():.4f} g")
print(f"   Grid points: {x_grid}")

PORTFOLIO DATA

1. EXPOSURE VECTOR (v) - Replacement costs in $
--------------------------------------------------------------------------------
   Asset 0: $  437,086.09
   Asset 1: $  955,642.88
   Asset 2: $  758,794.56
   Asset 3: $  638,792.62
   Asset 4: $  240,416.78
   Asset 5: $  240,395.06
   Asset 6: $  152,275.25
   Asset 7: $  879,558.50
   Asset 8: $  641,003.50
   Asset 9: $  737,265.31

   Total portfolio value: $5,681,231.00

2. TYPOLOGY ASSIGNMENT (u) - Which vulnerability curve each asset uses
--------------------------------------------------------------------------------
   Asset 0: Typology 4
   Asset 1: Typology 1
   Asset 2: Typology 3
   Asset 3: Typology 1
   Asset 4: Typology 3
   Asset 5: Typology 4
   Asset 6: Typology 0
   Asset 7: Typology 3
   Asset 8: Typology 1
   Asset 9: Typology 4

3. HAZARD INTENSITIES (h) - Ground motion in g for Event 0
--------------------------------------------------------------------------------
   Asset 0: 0.5183 g
   Asset 

In [4]:
print("\n5. VULNERABILITY MATRIX (C) - Mean Damage Ratio curves")
print("-" * 80)
print(f"   Shape: {C.shape} (5 typologies × 20 intensity points)")
print("\n   Typology 0 curve:")
print(f"   {C[0]}")
print("\n   Typology 1 curve:")
print(f"   {C[1]}")
print("\n   Typology 2 curve:")
print(f"   {C[2]}")
print("\n   Typology 3 curve:")
print(f"   {C[3]}")
print("\n   Typology 4 curve:")
print(f"   {C[4]}")


5. VULNERABILITY MATRIX (C) - Mean Damage Ratio curves
--------------------------------------------------------------------------------
   Shape: (5, 20) (5 typologies × 20 intensity points)

   Typology 0 curve:
   [0.03916572 0.07119864 0.12599519 0.21328057 0.33767235 0.48947528
 0.6432444  0.7722487  0.86443573 0.9230274  0.9575394  0.97696346
 0.9876167  0.9933768  0.9964671  0.9981183  0.99899846 0.9994672
 0.9997166  0.9998493 ]

   Typology 1 curve:
   [0.00669285 0.01462159 0.03164396 0.0671335  0.13680248 0.25872013
 0.4345876  0.62862337 0.7884803  0.8914136  0.94758564 0.9754984
 0.9887234  0.9948478  0.9976539  0.99893326 0.9995153  0.9997799
 0.9999     0.9999546 ]

   Typology 2 curve:
   [7.4602861e-04 1.9216796e-03 4.9408562e-03 1.2643413e-02 3.1968091e-02
 7.8481629e-02 1.8008237e-01 3.6160111e-01 5.9361905e-01 7.9023051e-01
 9.0667403e-01 9.6161884e-01 9.8475915e-01 9.9403459e-01 9.9767834e-01
 9.9909854e-01 9.9965024e-01 9.9986434e-01 9.9994743e-01 9.9997962e-01]



## Step 3: Manual Loss Calculation - Asset by Asset

Now we compute the loss for each asset using **explicit conventional logic** (loops and numpy operations).

### Mathematical Formulation

For each asset $i$:

1. **Get the hazard intensity**: $h_i$ (from hazard matrix)
2. **Get the typology**: $k = u_i$ (which vulnerability curve to use)
3. **Find grid interval**: Find $j$ such that $x_j \leq h_i < x_{j+1}$
4. **Compute interpolation weight**: $\alpha = \frac{h_i - x_j}{x_{j+1} - x_j}$
5. **Interpolate MDR**: $\text{MDR}_i = (1-\alpha) \cdot C[k, j] + \alpha \cdot C[k, j+1]$
6. **Compute loss**: $L_i = v_i \cdot \text{MDR}_i$

Finally: **Total Loss** = $\sum_{i=0}^{9} L_i$

In [5]:
print("="*80)
print("MANUAL LOSS CALCULATION - DETAILED STEP-BY-STEP")
print("="*80)

# Storage for results
asset_losses = np.zeros(10)
calculation_details = []

# Loop over each asset
for i in range(10):
    print(f"\n{'='*80}")
    print(f"ASSET {i}")
    print(f"{'='*80}")
    
    # Step 1: Get intensity
    intensity = h[i]
    print(f"\n1. Hazard intensity: h[{i}] = {intensity:.6f} g")
    
    # Step 2: Get typology
    typology = u[i]
    print(f"2. Typology: u[{i}] = {typology}")
    print(f"   → Will use vulnerability curve C[{typology}, :]")
    
    # Step 3: Get exposure
    exposure = v[i]
    print(f"3. Exposure: v[{i}] = ${exposure:,.2f}")
    
    # Step 4: Find the grid interval
    # We need to find j such that x_grid[j] <= intensity < x_grid[j+1]
    print(f"\n4. Finding grid interval for intensity {intensity:.6f} g:")
    
    # Use numpy searchsorted (same as TensorFlow's searchsorted)
    # 'right' gives the index where intensity would be inserted to maintain order
    # We subtract 1 to get the left boundary index
    j_right = np.searchsorted(x_grid, intensity, side='right')
    j = max(0, min(j_right - 1, len(x_grid) - 2))  # Clamp to valid range
    
    x_lower = x_grid[j]
    x_upper = x_grid[j + 1]
    
    print(f"   Grid point j = {j}")
    print(f"   Interval: [{x_lower:.6f}, {x_upper:.6f}] g")
    print(f"   Check: {x_lower:.6f} <= {intensity:.6f} < {x_upper:.6f}? ", end="")
    if x_lower <= intensity < x_upper or (j == len(x_grid) - 2 and intensity >= x_upper):
        print("✓ YES")
    else:
        print("⚠ BOUNDARY CASE")
    
    # Step 5: Compute interpolation weight
    denominator = x_upper - x_lower
    if denominator < 1e-8:
        alpha = 0.0
        print(f"\n5. Interpolation weight: α = 0.0 (denominator too small)")
    else:
        alpha = (intensity - x_lower) / denominator
        print(f"\n5. Interpolation weight:")
        print(f"   α = (h - x_j) / (x_{{j+1}} - x_j)")
        print(f"   α = ({intensity:.6f} - {x_lower:.6f}) / ({x_upper:.6f} - {x_lower:.6f})")
        print(f"   α = {intensity - x_lower:.6f} / {denominator:.6f}")
        print(f"   α = {alpha:.6f}")
    
    # Step 6: Get vulnerability values at grid points
    c_lower = C[typology, j]
    c_upper = C[typology, j + 1]
    print(f"\n6. Vulnerability values from curve C[{typology}]:")
    print(f"   C[{typology}, {j}] = {c_lower:.6f}")
    print(f"   C[{typology}, {j+1}] = {c_upper:.6f}")
    
    # Step 7: Interpolate MDR
    mdr = (1 - alpha) * c_lower + alpha * c_upper
    print(f"\n7. Interpolated Mean Damage Ratio (MDR):")
    print(f"   MDR = (1 - α) × C[{typology}, {j}] + α × C[{typology}, {j+1}]")
    print(f"   MDR = (1 - {alpha:.6f}) × {c_lower:.6f} + {alpha:.6f} × {c_upper:.6f}")
    print(f"   MDR = {(1-alpha):.6f} × {c_lower:.6f} + {alpha:.6f} × {c_upper:.6f}")
    print(f"   MDR = {(1-alpha)*c_lower:.6f} + {alpha*c_upper:.6f}")
    print(f"   MDR = {mdr:.6f}")
    
    # Step 8: Compute loss
    loss = exposure * mdr
    print(f"\n8. Asset loss:")
    print(f"   L[{i}] = v[{i}] × MDR")
    print(f"   L[{i}] = ${exposure:,.2f} × {mdr:.6f}")
    print(f"   L[{i}] = ${loss:,.2f}")
    
    # Store results
    asset_losses[i] = loss
    calculation_details.append({
        'asset': i,
        'intensity': intensity,
        'typology': typology,
        'exposure': exposure,
        'grid_index': j,
        'x_lower': x_lower,
        'x_upper': x_upper,
        'alpha': alpha,
        'c_lower': c_lower,
        'c_upper': c_upper,
        'mdr': mdr,
        'loss': loss
    })

print(f"\n{'='*80}")
print("CALCULATION COMPLETE")
print(f"{'='*80}")

MANUAL LOSS CALCULATION - DETAILED STEP-BY-STEP

ASSET 0

1. Hazard intensity: h[0] = 0.518334 g
2. Typology: u[0] = 4
   → Will use vulnerability curve C[4, :]
3. Exposure: v[0] = $437,086.09

4. Finding grid interval for intensity 0.518334 g:
   Grid point j = 6
   Interval: [0.473684, 0.552632] g
   Check: 0.473684 <= 0.518334 < 0.552632? ✓ YES

5. Interpolation weight:
   α = (h - x_j) / (x_{j+1} - x_j)
   α = (0.518334 - 0.473684) / (0.552632 - 0.473684)
   α = 0.044650 / 0.078947
   α = 0.565565

6. Vulnerability values from curve C[4]:
   C[4, 6] = 0.005373
   C[4, 7] = 0.018745

7. Interpolated Mean Damage Ratio (MDR):
   MDR = (1 - α) × C[4, 6] + α × C[4, 7]
   MDR = (1 - 0.565565) × 0.005373 + 0.565565 × 0.018745
   MDR = 0.434435 × 0.005373 + 0.565565 × 0.018745
   MDR = 0.002334 + 0.010602
   MDR = 0.012936

8. Asset loss:
   L[0] = v[0] × MDR
   L[0] = $437,086.09 × 0.012936
   L[0] = $5,654.00

ASSET 1

1. Hazard intensity: h[1] = 0.349475 g
2. Typology: u[1] = 1
   → Wil

## Step 4: Summary of Results

Display all asset losses and compute total portfolio loss.

In [6]:
print("="*80)
print("SUMMARY TABLE - ALL ASSETS")
print("="*80)
print(f"\n{'Asset':<6} {'Intensity':<12} {'Type':<5} {'Exposure':<15} {'MDR':<10} {'Loss':<15}")
print("-" * 80)

for detail in calculation_details:
    print(f"{detail['asset']:<6} "
          f"{detail['intensity']:<12.6f} "
          f"{detail['typology']:<5} "
          f"${detail['exposure']:<14,.2f} "
          f"{detail['mdr']:<10.6f} "
          f"${detail['loss']:<14,.2f}")

# Compute total loss
total_loss_manual = asset_losses.sum()

print("-" * 80)
print(f"{'TOTAL':<6} {'':<12} {'':<5} ${v.sum():<14,.2f} {'':<10} ${total_loss_manual:<14,.2f}")
print("=" * 80)

print(f"\n📊 MANUAL CALCULATION RESULT:")
print(f"   Total Portfolio Loss = ${total_loss_manual:,.2f}")

SUMMARY TABLE - ALL ASSETS

Asset  Intensity    Type  Exposure        MDR        Loss           
--------------------------------------------------------------------------------
0      0.518334     4     $437,086.09     0.012936   $5,654.00      
1      0.349475     1     $955,642.88     0.188823   $180,447.03    
2      0.734223     3     $758,794.56     0.609109   $462,188.60    
3      0.167393     1     $638,792.62     0.035914   $22,941.33     
4      0.350574     3     $240,416.78     0.008622   $2,072.84      
5      0.439634     4     $240,395.06     0.003713   $892.62        
6      0.547284     0     $152,275.25     0.763510   $116,263.75    
7      0.942211     3     $879,558.50     0.965959   $849,617.30    
8      0.239609     1     $641,003.50     0.069575   $44,597.69     
9      0.617081     4     $737,265.31     0.055106   $40,627.42     
--------------------------------------------------------------------------------
TOTAL                     $5,681,231.00            

## Step 5: Validation Against Tensor Engine

Now let's compute the same loss using the tensorial engine and compare results.

In [7]:
# Use the tensorial engine's deterministic_loss function
v_tf = tf.constant(v, dtype=tf.float32)
u_tf = tf.constant(u, dtype=tf.int32)
C_tf = tf.constant(C, dtype=tf.float32)
x_tf = tf.constant(x_grid, dtype=tf.float32)
h_tf = tf.constant(h, dtype=tf.float32)

# Compute loss using tensor engine
loss_tensor = deterministic_loss(v_tf, u_tf, C_tf, x_tf, h_tf)

print("="*80)
print("TENSOR ENGINE RESULT")
print("="*80)
print(f"\nTotal Portfolio Loss (Tensor Engine) = ${loss_tensor.numpy():,.2f}")

# Comparison
print("\n" + "="*80)
print("COMPARISON")
print("="*80)
print(f"\nManual calculation:    ${total_loss_manual:,.2f}")
print(f"Tensor engine:         ${loss_tensor.numpy():,.2f}")
print(f"Absolute difference:   ${abs(total_loss_manual - loss_tensor.numpy()):,.2f}")
print(f"Relative difference:   {abs(total_loss_manual - loss_tensor.numpy()) / total_loss_manual * 100:.6f}%")

if abs(total_loss_manual - loss_tensor.numpy()) < 1e-2:
    print("\n✅ VALIDATION SUCCESSFUL - Results match within numerical precision!")
else:
    print("\n⚠️ WARNING - Results differ by more than expected!")

2026-02-17 15:04:52.218559: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-02-17 15:04:52.219109: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-17 15:04:52.219129: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-02-17 15:04:52.219715: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-17 15:04:52.219761: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-02-17 15:04:52.357932: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


TENSOR ENGINE RESULT

Total Portfolio Loss (Tensor Engine) = $1,725,302.62

COMPARISON

Manual calculation:    $1,725,302.59
Tensor engine:         $1,725,302.62
Absolute difference:   $0.04
Relative difference:   0.000002%

⚠️ WARNING - Results differ by more than expected!


## Step 6: Detailed Comparison Table

Let's create a detailed comparison showing each asset's loss from both methods.

In [8]:
# Compute per-asset losses using the tensor engine
# We'll compute this by running the engine with just this one event
engine = TensorialRiskEngine(v, u, C, x_grid, H)
J_matrix, metrics = engine.compute_loss_and_metrics()

# Extract per-asset losses for the single event
tensor_asset_losses = J_matrix[:, 0].numpy()

print("="*80)
print("PER-ASSET COMPARISON")
print("="*80)
print(f"\n{'Asset':<6} {'Manual Loss':<20} {'Tensor Loss':<20} {'Difference':<15}")
print("-" * 80)

for i in range(10):
    diff = abs(asset_losses[i] - tensor_asset_losses[i])
    print(f"{i:<6} ${asset_losses[i]:<19,.2f} ${tensor_asset_losses[i]:<19,.2f} ${diff:<14,.6f}")

print("-" * 80)
print(f"{'TOTAL':<6} ${asset_losses.sum():<19,.2f} ${tensor_asset_losses.sum():<19,.2f} "
      f"${abs(asset_losses.sum() - tensor_asset_losses.sum()):<14,.6f}")
print("=" * 80)

PER-ASSET COMPARISON

Asset  Manual Loss          Tensor Loss          Difference     
--------------------------------------------------------------------------------
0      $5,654.00            $5,654.00            $0.000181      
1      $180,447.03          $180,447.03          $0.002906      
2      $462,188.60          $462,188.62          $0.024269      
3      $22,941.33           $22,941.33           $0.000568      
4      $2,072.84            $2,072.84            $0.000100      
5      $892.62              $892.62              $0.000021      
6      $116,263.75          $116,263.75          $0.001047      
7      $849,617.30          $849,617.31          $0.012880      
8      $44,597.69           $44,597.69           $0.001376      
9      $40,627.42           $40,627.42           $0.001850      
--------------------------------------------------------------------------------
TOTAL  $1,725,302.59        $1,725,302.62        $0.035445      


## Step 7: Display Key Quantities for Verification

Export all intermediate values that can be used for hand-checking calculations.

In [9]:
print("="*80)
print("VERIFICATION DATA - ALL INTERMEDIATE QUANTITIES")
print("="*80)

for i, detail in enumerate(calculation_details):
    print(f"\n{'='*80}")
    print(f"ASSET {i} - COMPLETE CALCULATION RECORD")
    print(f"{'='*80}")
    print(f"Input Parameters:")
    print(f"  • Exposure (v[{i}]):          ${detail['exposure']:,.2f}")
    print(f"  • Typology (u[{i}]):          {detail['typology']}")
    print(f"  • Intensity (h[{i}]):         {detail['intensity']:.8f} g")
    print(f"\nGrid Interpolation:")
    print(f"  • Grid index (j):           {detail['grid_index']}")
    print(f"  • Lower bound (x[{detail['grid_index']}]):     {detail['x_lower']:.8f} g")
    print(f"  • Upper bound (x[{detail['grid_index']+1}]):     {detail['x_upper']:.8f} g")
    print(f"  • Interval width (Δx):      {detail['x_upper'] - detail['x_lower']:.8f} g")
    print(f"\nInterpolation:")
    print(f"  • Weight (α):               {detail['alpha']:.8f}")
    print(f"  • 1 - α:                    {1 - detail['alpha']:.8f}")
    print(f"\nVulnerability Values:")
    print(f"  • C[{detail['typology']}, {detail['grid_index']}]:              {detail['c_lower']:.8f}")
    print(f"  • C[{detail['typology']}, {detail['grid_index']+1}]:              {detail['c_upper']:.8f}")
    print(f"\nMean Damage Ratio:")
    print(f"  • MDR = (1-α)×C_lower + α×C_upper")
    print(f"  • MDR = {1-detail['alpha']:.8f} × {detail['c_lower']:.8f} + {detail['alpha']:.8f} × {detail['c_upper']:.8f}")
    print(f"  • MDR = {detail['mdr']:.8f}")
    print(f"\nLoss Computation:")
    print(f"  • Loss = Exposure × MDR")
    print(f"  • Loss = ${detail['exposure']:,.2f} × {detail['mdr']:.8f}")
    print(f"  • Loss = ${detail['loss']:,.2f}")
    print(f"\nVerification:")
    print(f"  • Manual loss:              ${detail['loss']:,.2f}")
    print(f"  • Tensor engine loss:       ${tensor_asset_losses[i]:,.2f}")
    print(f"  • Absolute difference:      ${abs(detail['loss'] - tensor_asset_losses[i]):.6f}")

print(f"\n{'='*80}")
print("FINAL TOTALS")
print(f"{'='*80}")
print(f"Manual calculation total:    ${total_loss_manual:,.2f}")
print(f"Tensor engine total:         ${loss_tensor.numpy():,.2f}")
print(f"Absolute difference:         ${abs(total_loss_manual - loss_tensor.numpy()):.6f}")
print(f"Relative error:              {abs(total_loss_manual - loss_tensor.numpy()) / total_loss_manual * 100:.8f}%")

VERIFICATION DATA - ALL INTERMEDIATE QUANTITIES

ASSET 0 - COMPLETE CALCULATION RECORD
Input Parameters:
  • Exposure (v[0]):          $437,086.09
  • Typology (u[0]):          4
  • Intensity (h[0]):         0.51833403 g

Grid Interpolation:
  • Grid index (j):           6
  • Lower bound (x[6]):     0.47368422 g
  • Upper bound (x[7]):     0.55263156 g
  • Interval width (Δx):      0.07894734 g

Interpolation:
  • Weight (α):               0.56556451
  • 1 - α:                    0.43443549

Vulnerability Values:
  • C[4, 6]:              0.00537262
  • C[4, 7]:              0.01874518

Mean Damage Ratio:
  • MDR = (1-α)×C_lower + α×C_upper
  • MDR = 0.43443549 × 0.00537262 + 0.56556451 × 0.01874518
  • MDR = 0.01293567

Loss Computation:
  • Loss = Exposure × MDR
  • Loss = $437,086.09 × 0.01293567
  • Loss = $5,654.00

Verification:
  • Manual loss:              $5,654.00
  • Tensor engine loss:       $5,654.00
  • Absolute difference:      $0.000181

ASSET 1 - COMPLETE CALCULATION

## Conclusion

This notebook demonstrates the complete hand calculation process for the deterministic loss computation. 

### What we validated:

✅ **Grid-based interpolation** - Finding the correct interval for each intensity  
✅ **Linear interpolation** - Computing interpolation weights and MDR values  
✅ **Loss aggregation** - Summing individual asset losses  
✅ **Numerical consistency** - Manual calculation matches tensor engine

### Key observations:

1. Each asset's loss depends on: exposure, typology, and hazard intensity
2. The interpolation is piecewise-linear on the intensity grid
3. Different typologies have different vulnerability curves
4. The tensor engine reproduces exact manual calculations (within floating-point precision)

This validates that the tensorial formulation correctly implements the mathematical model described in the manuscript.